In [1]:
"""
Problem 01: Top-N per group (e.g., top 3 posts per user by engagement).

Meta flavor: "Show the top 5 reels by watch-time per creator in the last 7 days."

How to Think:
- Rank within a partition with ROW_NUMBER() or RANK(), then filter.
- ROW_NUMBER() = exact N per group; RANK() = allow ties (may exceed N).
- Always partition by the group key; order by the metric descending.

How to Remember:
- "PARTITION BY group, ORDER BY metric DESC, then wrap and filter rn <= N."
- ROW_NUMBER breaks ties arbitrarily → use when duplicates are not meaningful.

AI Use Cases:
- Top-k recommendation candidates before reranking.
- Per-user feature extraction (top 3 clicked items → embedding).
- Anomaly detection: top-1 baseline vs current.
"""
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

spark = SparkSession.builder.getOrCreate()

# posts(user_id, post_id, engagement_score, created_at)
data = [
    (1, 101, 50, "2026-01-01"),
    (1, 102, 80, "2026-01-02"),
    (1, 103, 30, "2026-01-03"),
    (2, 201, 90, "2026-01-01"),
    (2, 202, 70, "2026-01-02"),
]
posts = spark.createDataFrame(data, ["user_id", "post_id", "engagement_score", "created_at"])

w = Window.partitionBy("user_id").orderBy(col("engagement_score").desc())
top3 = posts.withColumn("rn", row_number().over(w)).filter(col("rn") <= 3)
top3.show()
posts.createOrReplaceTempView("user_data_vw")
# SQL equivalent (Presto / Hive)
SQL = """
select * from 
(
select user_id,post_id,engagement_score,created_at , ROW_NUMBER() OVER (partition by user_id order by  engagement_score DESC ) as rn_num 
FROM user_data_vw
) t
WHERE rn_num <= 1;
"""

posts.createOrReplaceTempView("user_data_vw")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/27 15:20:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+-------+-------+----------------+----------+---+
|user_id|post_id|engagement_score|created_at| rn|
+-------+-------+----------------+----------+---+
|      1|    102|              80|2026-01-02|  1|
|      1|    101|              50|2026-01-01|  2|
|      1|    103|              30|2026-01-03|  3|
|      2|    201|              90|2026-01-01|  1|
|      2|    202|              70|2026-01-02|  2|
+-------+-------+----------------+----------+---+



In [2]:
spark.sql(SQL).show()

+-------+-------+----------------+----------+------+
|user_id|post_id|engagement_score|created_at|rn_num|
+-------+-------+----------------+----------+------+
|      1|    102|              80|2026-01-02|     1|
|      2|    201|              90|2026-01-01|     1|
+-------+-------+----------------+----------+------+



In [3]:
spark.sql("select * from user_data_vw").show()

+-------+-------+----------------+----------+
|user_id|post_id|engagement_score|created_at|
+-------+-------+----------------+----------+
|      1|    101|              50|2026-01-01|
|      1|    102|              80|2026-01-02|
|      1|    103|              30|2026-01-03|
|      2|    201|              90|2026-01-01|
|      2|    202|              70|2026-01-02|
+-------+-------+----------------+----------+



In [4]:
"""
Problem 02: Compare current session to previous session using LAG / LEAD.

Meta flavor: "For each user, compute the time delta between consecutive sessions."

How to Think:
- LAG(col, 1) reads the previous row's value within the same partition.
- LEAD(col, 1) reads the next row's value.
- Order by event_time per user to define "consecutive."

How to Remember:
- "LAG = look BACK, LEAD = look FORWARD."
- Always order inside the window or it's meaningless.

AI Use Cases:
- Sequential feature engineering (delta time → RNN input).
- Churn prediction: gap between sessions.
- Drop-off detection: long delays before next step.
"""
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import col, lag, unix_timestamp

spark = SparkSession.builder.getOrCreate()

data = [
    (1, "session_start", "2026-01-01 10:00:00"),
    (1, "session_start", "2026-01-01 14:30:00"),
    (1, "session_start", "2026-01-03 09:00:00"),
    (2, "session_start", "2026-01-02 11:00:00"),
]
events = spark.createDataFrame(data, ["user_id", "event", "ts"])

w = Window.partitionBy("user_id").orderBy("ts")
result = events.withColumn("prev_ts", lag("ts", 1).over(w))
result = result.withColumn(
    "delta_hours",
    (unix_timestamp("ts") - unix_timestamp("prev_ts")) / 3600
)
result.show(truncate=False)

SQL = """
SELECT user_id, event, ts,
       LAG(ts, 1) OVER (PARTITION BY user_id ORDER BY ts) AS prev_ts,
       UNIX_TIMESTAMP(ts) - UNIX_TIMESTAMP(LAG(ts, 1) OVER (PARTITION BY user_id ORDER BY ts)) AS delta_sec
FROM events;
"""


+-------+-------------+-------------------+-------------------+-----------+
|user_id|event        |ts                 |prev_ts            |delta_hours|
+-------+-------------+-------------------+-------------------+-----------+
|1      |session_start|2026-01-01 10:00:00|NULL               |NULL       |
|1      |session_start|2026-01-01 14:30:00|2026-01-01 10:00:00|4.5        |
|1      |session_start|2026-01-03 09:00:00|2026-01-01 14:30:00|42.5       |
|2      |session_start|2026-01-02 11:00:00|NULL               |NULL       |
+-------+-------------+-------------------+-------------------+-----------+



In [19]:
events.createOrReplaceTempView("event_vw")

In [20]:
spark.sql("select * from event_vw").show()

+-------+-------------+-------------------+
|user_id|        event|                 ts|
+-------+-------------+-------------------+
|      1|session_start|2026-01-01 10:00:00|
|      1|session_start|2026-01-01 14:30:00|
|      1|session_start|2026-01-03 09:00:00|
|      2|session_start|2026-01-02 11:00:00|
+-------+-------------+-------------------+



In [5]:
SQL = """
SELECT user_id, event, ts,
       LAG(ts, 1) OVER (PARTITION BY user_id ORDER BY ts) AS prev_ts,
       (UNIX_TIMESTAMP(ts) - UNIX_TIMESTAMP(LAG(ts, 1) OVER (PARTITION BY user_id ORDER BY ts)))/(60*60) AS delta_sec
FROM event_vw;
"""


In [7]:
# spark.sql(SQL).show()

In [8]:
print(1)
# spark.stop()

1


In [9]:
"""
Problem 01: D1 retention by signup cohort.

Meta flavor: "What share of users who signed up on a given day came back the
next day?"

How to Think:
- Retention is ALWAYS (cohort, day-offset). Name both out loud before any SQL.
- "D1" = active on exactly signup_date + 1 (bounded / classic retention).
- The D1 target date is DIFFERENT FOR EVERY USER, because it is relative to
  that user's own signup. u1 signed up 01-01 so their D1 is 01-02; u4 signed up
  01-02 so their D1 is 01-03. That is why the join computes
  DATE_ADD(u.signup_date, 1) instead of comparing against one fixed date.
- Denominator = cohort size, NOT total users. This is the #1 error.
- LEFT JOIN the activity, never INNER. An INNER JOIN drops churned users from
  the denominator too, and retention comes out as 100% every time.

Why DATE_ADD on the USERS side and not DATEDIFF on the events side:
    GOOD:  e.event_date = DATE_ADD(u.signup_date, 1)
    BAD:   DATEDIFF(e.event_date, u.signup_date) = 1
  Both are logically identical. The first leaves `e.event_date` as a bare
  column, so a partitioned events table can still prune. The second wraps the
  events column in a function, which kills partition pruning and forces a full
  scan. Meta asks "would this scan the whole table?" — this is that answer.

The integer-division trap:
- In PRESTO, COUNT(...)/COUNT(...) on BIGINTs is integer division and returns 0.
  Multiply by 100.0 (a decimal literal) to force decimal arithmetic.
  (Spark and Hive return a double here, so the trap is Presto-specific — but
  Meta runs Presto, so always write the 100.0.)

AI Use Cases:
- Early-churn labels for a propensity model.
- Cohort features for LTV regression.
"""
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .appName("d1-retention")
         .master("local[2]")
         .config("spark.sql.shuffle.partitions", "2")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

# ---------------------------------------------------------------- sample data
# users(user_id, signup_date) — the cohort table.
users_data = [
    (1, "2026-01-01"),
    (2, "2026-01-01"),
    (3, "2026-01-01"),
    (4, "2026-01-02"),
    (5, "2026-01-02"),
    (6, "2026-01-02"),
    (7, "2026-01-08"),
    (8, "2026-01-08"),
]
users = spark.createDataFrame(users_data, ["user_id", "signup_date"])
users.createOrReplaceTempView("users")

# events(user_id, event_date, event_name) — the activity table.
# Designed so each cohort has at least one churned user:
#   u1 D0,D1,D7   u2 D0 only(churn)  u3 D0,D1
#   u4 D0,D1,D7   u5 D0 only(churn)  u6 D0,D1,D28
#   u7 D0 only(churn)                u8 D0,D1
events_data = [
    (1, "2026-01-01", "open"), (1, "2026-01-02", "open"), (1, "2026-01-08", "open"),
    (2, "2026-01-01", "open"),
    (3, "2026-01-01", "open"), (3, "2026-01-02", "open"),
    (4, "2026-01-02", "open"), (4, "2026-01-03", "open"), (4, "2026-01-09", "open"),
    (5, "2026-01-02", "open"),
    (6, "2026-01-02", "open"), (6, "2026-01-03", "open"), (6, "2026-01-30", "open"),
    (7, "2026-01-08", "open"),
    (8, "2026-01-08", "open"), (8, "2026-01-09", "open"),
]
events = spark.createDataFrame(events_data, ["user_id", "event_date", "event_name"])
events.createOrReplaceTempView("events")

# ------------------------------------------- show the per-user logic first
# Always eyeball this before trusting the aggregate. Note d1_target changes
# per user, and that d1_hit uses an INT flag — MAX() over the strings
# 'YES'/'no' would return 'no' every time, because 'n' > 'Y' in ASCII.
print("\n--- per-user D1 check (verify the aggregate against this) ---")
spark.sql("""
SELECT u.user_id, u.signup_date,
       DATE_ADD(u.signup_date, 1) AS d1_target,
       CONCAT_WS(',', SORT_ARRAY(COLLECT_LIST(e.event_date))) AS active_dates,
       MAX(CASE WHEN e.event_date = DATE_ADD(u.signup_date, 1) THEN 1 ELSE 0 END) AS d1_hit
FROM users u
LEFT JOIN events e ON e.user_id = u.user_id
GROUP BY u.user_id, u.signup_date
ORDER BY u.signup_date, u.user_id
""").show(truncate=False)

# ---------------------------------------------------------------- SQL solution
SQL = """
SELECT u.signup_date AS cohort,
       COUNT(DISTINCT u.user_id) AS cohort_size,
       COUNT(DISTINCT e.user_id) AS retained_d1,
       ROUND(100.0 * COUNT(DISTINCT e.user_id) / COUNT(DISTINCT u.user_id), 2) AS pct_d1
FROM users u
LEFT JOIN events e
       ON e.user_id = u.user_id
      AND e.event_date = DATE_ADD(u.signup_date, 1)
GROUP BY u.signup_date
ORDER BY u.signup_date
"""

print("--- D1 retention by cohort (SQL) ---")
result = spark.sql(SQL)
result.show(truncate=False)

EXPECTED = [
    ("2026-01-01", 3, 2, 66.67),
    ("2026-01-02", 3, 2, 66.67),
    ("2026-01-08", 2, 1, 50.00),
]
got = [(r[0], r[1], r[2], float(r[3])) for r in result.collect()]
assert got == EXPECTED, f"\nexpected {EXPECTED}\ngot      {got}"
print("[PASS] D1 retention by cohort (SQL)")

# ------------------------------------------------- PySpark DataFrame API
# Same plan: build the D1 target on the cohort side, LEFT JOIN, count distinct.
df = (users
      .withColumn("d1_target", F.date_add(F.col("signup_date"), 1))
      .join(events.alias("e"),
            (F.col("e.user_id") == users["user_id"])
            & (F.col("e.event_date") == F.col("d1_target")),
            "left")
      .groupBy(users["signup_date"].alias("cohort"))
      .agg(F.countDistinct(users["user_id"]).alias("cohort_size"),
           F.countDistinct(F.col("e.user_id")).alias("retained_d1"))
      .withColumn("pct_d1",
                  F.round(100.0 * F.col("retained_d1") / F.col("cohort_size"), 2))
      .orderBy("cohort"))

print("--- D1 retention by cohort (DataFrame API) ---")
df.show(truncate=False)
got_df = [(r[0], r[1], r[2], float(r[3])) for r in df.collect()]
assert got_df == EXPECTED, f"\nexpected {EXPECTED}\ngot      {got_df}"
print("[PASS] D1 retention — DataFrame API matches SQL")

# --------------------------------------------- the INNER JOIN failure mode
# Asserted so the bug is documented: churned users vanish from the DENOMINATOR
# too, so every cohort reports 100% retention.
print("--- WRONG: INNER JOIN inflates retention to 100% ---")
wrong = spark.sql("""
SELECT u.signup_date AS cohort,
       COUNT(DISTINCT u.user_id) AS cohort_size,
       COUNT(DISTINCT e.user_id) AS retained_d1,
       ROUND(100.0 * COUNT(DISTINCT e.user_id) / COUNT(DISTINCT u.user_id), 2) AS pct_d1
FROM users u
JOIN events e ON e.user_id = u.user_id
             AND e.event_date = DATE_ADD(u.signup_date, 1)
GROUP BY u.signup_date ORDER BY u.signup_date
""")
wrong.show(truncate=False)
got_wrong = [(r[0], r[1], r[2], float(r[3])) for r in wrong.collect()]
assert got_wrong == [
    ("2026-01-01", 2, 2, 100.00),
    ("2026-01-02", 2, 2, 100.00),
    ("2026-01-08", 1, 1, 100.00),
], got_wrong
print("[PASS] INNER JOIN failure mode reproduced (cohort_size shrank: 3->2, 3->2, 2->1)")



26/09/27 15:25:26 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.



--- per-user D1 check (verify the aggregate against this) ---
+-------+-----------+----------+--------------------------------+------+
|user_id|signup_date|d1_target |active_dates                    |d1_hit|
+-------+-----------+----------+--------------------------------+------+
|1      |2026-01-01 |2026-01-02|2026-01-01,2026-01-02,2026-01-08|1     |
|2      |2026-01-01 |2026-01-02|2026-01-01                      |0     |
|3      |2026-01-01 |2026-01-02|2026-01-01,2026-01-02           |1     |
|4      |2026-01-02 |2026-01-03|2026-01-02,2026-01-03,2026-01-09|1     |
|5      |2026-01-02 |2026-01-03|2026-01-02                      |0     |
|6      |2026-01-02 |2026-01-03|2026-01-02,2026-01-03,2026-01-30|1     |
|7      |2026-01-08 |2026-01-09|2026-01-08                      |0     |
|8      |2026-01-08 |2026-01-09|2026-01-08,2026-01-09           |1     |
+-------+-----------+----------+--------------------------------+------+

--- D1 retention by cohort (SQL) ---
+----------+-----------

In [10]:
"""
Problem 02: D7 retention by signup cohort.

Meta flavor: "D7 retention for the January 1 cohort."

Business Question
-----------------
Of the users who signed up on each day, how many came back exactly seven
days later? This is the canonical "early-habit" KPI for consumer products:
D1 measures onboarding, D7 measures whether the product has earned a second
visit, D28 measures habit.

How to Think
------------
- Retention is ALWAYS (cohort, day-offset). Say both out loud before writing
  SQL or you will silently produce the wrong denominator.
- "D7" at Meta = active on exactly signup_date + 7 (bounded / classic
  retention), not "active any time during days 1..7" (rolling retention).
- Denominator = cohort size, NOT total users. Mixing them up is the #1 bug.
- LEFT JOIN the activity, never INNER — an INNER JOIN silently drops churned
  users and inflates retention to 100%.
- Watch the reporting window: a cohort younger than 7 days CANNOT have D7
  data. Reporting 0% for an immature cohort is a real bug — exclude it
  instead. Here the 2026-01-08 cohort genuinely has no D7 activity in the
  seed data.
- D7 differs from D1 only by the offset (7 vs 1). Parameterise the offset
  rather than writing seven near-identical queries (see
  06_retention_curve_d0_d7.py).

How to Remember
---------------
"Same skeleton, new offset. Guard the immature cohorts."

The Integer-Division Trap
-------------------------
COUNT(...) / COUNT(...) is integer division in Presto/Hive and returns 0.
Multiply by 100.0 (a decimal literal) to force floating point. Spark is
slightly more forgiving than Presto here, but write it defensively anyway
because the same query will run on both engines at Meta.

Spark-Native Rewrite (CTE + window-friendly aggregation)
-------------------------------------------------------
The classic LEFT JOIN form duplicates the cohort_size denominator next to
the retained numerator. In this rewrite we split the work into stages:

    1. `cohorts`     CTE  — one row per signup_date with cohort_size
                            pre-computed and the literal `signup_date + 7`
                            marker cached for the join below.
    2. `d7_activity` CTE  — one row per (cohort, retained_user) whose
                            event_date lands exactly on the D7 marker.
                            DISTINCT inside the CTE keeps the join from
                            blowing up when a user fires multiple events
                            on day +7.
    3. Outer query        — LEFT JOIN cohorts -> d7_activity so churned
                            users survive, then aggregate with
                            COUNT(DISTINCT) and divide against the
                            pre-computed cohort_size. No integer division
                            because 100.0 * COUNT(DISTINCT ...) is a
                            double literal.

The window-function alternative (COUNT(...) OVER (PARTITION BY cohort))
is shown as a comment below the SQL for reviewers who prefer that style.

AI Use Cases
------------
- D7 is the standard early-retention target variable for growth models
  (churn-propensity classifier, lifecycle-stage regressor).
- Cohort × offset matrix is a feature block for LTV regression.
- D7 lift is the primary north-star metric in most onboarding A/B tests.
"""
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName("02-d7-retention")
         .master("local[2]")
         .config("spark.sql.shuffle.partitions", "2")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")


def expect(title, sql, expected_rows):
    """Run a query and assert its exact rows, in order. Decimal/float safe."""
    import decimal

    def norm(v):
        if isinstance(v, decimal.Decimal):
            return float(v)
        if isinstance(v, float):
            return round(v, 6)
        return v

    got = [tuple(norm(c) for c in r) for r in spark.sql(sql).collect()]
    exp = [tuple(norm(c) for c in r) for r in expected_rows]
    if got != exp:
        print(f"[FAIL] {title}")
        print(f"   expected: {exp}")
        print(f"   got:      {got}")
        raise AssertionError(title)
    print(f"[PASS] {title}")
    return got


# ---------------------------------------------------------- sample data
spark.createDataFrame(
    [
    (1, "2026-01-01", "organic",  "US"),
    (2, "2026-01-01", "paid",     "US"),
    (3, "2026-01-01", "organic",  "IN"),
    (4, "2026-01-02", "paid",     "US"),
    (5, "2026-01-02", "referral", "BR"),
    (6, "2026-01-02", "organic",  "IN"),
    (7, "2026-01-08", "paid",     "US"),
    (8, "2026-01-08", "organic",  "BR"),
],
    ["user_id", "signup_date", "channel", "country"]
).createOrReplaceTempView("users")

spark.createDataFrame(
    [
    (1, "2026-01-01", "open"),  (1, "2026-01-02", "open"),  (1, "2026-01-08", "open"),
    (2, "2026-01-01", "open"),
    (3, "2026-01-01", "open"),  (3, "2026-01-02", "open"),
    (4, "2026-01-02", "open"),  (4, "2026-01-03", "open"),  (4, "2026-01-09", "open"),
    (5, "2026-01-02", "open"),
    (6, "2026-01-02", "open"),  (6, "2026-01-03", "open"),  (6, "2026-01-30", "open"),
    (7, "2026-01-08", "open"),
    (8, "2026-01-08", "open"),  (8, "2026-01-09", "open"),
],
    ["user_id", "event_date", "event_name"]
).createOrReplaceTempView("events")


SQL = """
WITH cohorts AS (
    -- One row per signup cohort with its size pre-computed.
    SELECT signup_date                                        AS cohort,
           COUNT(DISTINCT user_id)                            AS cohort_size,
           MAX(DATE_ADD(signup_date, 7))                      AS d7_marker
    FROM users
    GROUP BY signup_date
),
d7_activity AS (
    -- One row per user who was active exactly 7 days after signup.
    SELECT DISTINCT
           u.signup_date                                      AS cohort,
           u.user_id                                          AS retained_user
    FROM users u
    JOIN events e
      ON e.user_id    = u.user_id
     AND e.event_date = DATE_ADD(u.signup_date, 7)
)
SELECT c.cohort                                                    AS cohort,
       c.cohort_size                                               AS cohort_size,
       COUNT(DISTINCT a.retained_user)                             AS retained_d7,
       ROUND(
           100.0 * COUNT(DISTINCT a.retained_user) / c.cohort_size,
           2
       )                                                           AS pct_d7
FROM cohorts c
LEFT JOIN d7_activity a
       ON a.cohort = c.cohort
GROUP BY c.cohort, c.cohort_size
ORDER BY c.cohort
"""

# Alternative compact form using a window function over the joined set:
#
# SELECT cohort,
#        MAX(cohort_size)                                            AS cohort_size,
#        COUNT(DISTINCT CASE WHEN is_active_d7 THEN user_id END)     AS retained_d7,
#        ROUND(
#            100.0 * COUNT(DISTINCT CASE WHEN is_active_d7 THEN user_id END)
#                  / MAX(cohort_size),
#            2
#        )                                                          AS pct_d7
# FROM (
#     SELECT u.signup_date                                           AS cohort,
#            u.user_id,
#            COUNT(DISTINCT u.user_id)                               OVER (PARTITION BY u.signup_date)
#                                                                     AS cohort_size,
#            MAX(CASE WHEN e.event_date = DATE_ADD(u.signup_date, 7)
#                     THEN 1 ELSE 0 END)                             AS is_active_d7
#     FROM users u
#     LEFT JOIN events e ON e.user_id = u.user_id
#     GROUP BY u.signup_date, u.user_id
# ) t
# GROUP BY cohort
# ORDER BY cohort;

expect("D7 retention by cohort", SQL, [
    ("2026-01-01", 3, 1, 33.33),
    ("2026-01-02", 3, 1, 33.33),
    ("2026-01-08", 2, 0, 0.00),
])


[PASS] D7 retention by cohort


[('2026-01-01', 3, 1, 33.33),
 ('2026-01-02', 3, 1, 33.33),
 ('2026-01-08', 2, 0, 0.0)]

In [14]:
"""
Problem 03: D28 retention by signup cohort.

Meta flavor: "D28 retention — does the product hold users for a month?"

How to Think:
- Same skeleton again. The interesting part is what D28 means for the business:
  D1 measures onboarding, D28 measures habit.
- Only user 6 (signup 2026-01-02, active 2026-01-30) hits D28 in this seed, which
  is exactly the kind of sparse tail you see in real cohort tables.

How to Remember:
- "D1 = onboarding. D7 = interest. D28 = habit."

AI Use Cases:
- D28 is the usual 'good user' label for acquisition-quality scoring.
"""
# from pyspark.sql import SparkSession

# spark = (SparkSession.builder
#          .appName("03-d28-retention")
#          .master("local[2]")
#          .config("spark.sql.shuffle.partitions", "2")
#          .config("spark.ui.showConsoleProgress", "false")
#          .getOrCreate())
# spark.sparkContext.setLogLevel("ERROR")


def expect(title, sql, expected_rows):
    """Run a query and assert its exact rows, in order. Decimal/float safe."""
    import decimal

    def norm(v):
        if isinstance(v, decimal.Decimal):
            return float(v)
        if isinstance(v, float):
            return round(v, 6)
        return v

    got = [tuple(norm(c) for c in r) for r in spark.sql(sql).collect()]
    exp = [tuple(norm(c) for c in r) for r in expected_rows]
    if got != exp:
        print(f"[FAIL] {title}")
        print(f"   expected: {exp}")
        print(f"   got:      {got}")
        raise AssertionError(title)
    print(f"[PASS] {title}")
    return got


# ---------------------------------------------------------- sample data
spark.createDataFrame(
    [
    (1, "2026-01-01", "organic",  "US"),
    (2, "2026-01-01", "paid",     "US"),
    (3, "2026-01-01", "organic",  "IN"),
    (4, "2026-01-02", "paid",     "US"),
    (5, "2026-01-02", "referral", "BR"),
    (6, "2026-01-02", "organic",  "IN"),
    (7, "2026-01-08", "paid",     "US"),
    (8, "2026-01-08", "organic",  "BR"),
],
    ["user_id", "signup_date", "channel", "country"]
).createOrReplaceTempView("users")

spark.createDataFrame(
    [
    (1, "2026-01-01", "open"),  (1, "2026-01-02", "open"),  (1, "2026-01-08", "open"),
    (2, "2026-01-01", "open"),
    (3, "2026-01-01", "open"),  (3, "2026-01-02", "open"),
    (4, "2026-01-02", "open"),  (4, "2026-01-03", "open"),  (4, "2026-01-09", "open"),
    (5, "2026-01-02", "open"),
    (6, "2026-01-02", "open"),  (6, "2026-01-03", "open"),  (6, "2026-01-30", "open"),
    (7, "2026-01-08", "open"),
    (8, "2026-01-08", "open"),  (8, "2026-01-09", "open"),
],
    ["user_id", "event_date", "event_name"]
).createOrReplaceTempView("events")


SQL = """
SELECT u.signup_date AS cohort,
       COUNT(DISTINCT u.user_id) AS cohort_size,
       COUNT(DISTINCT e.user_id) AS retained_d28,
       ROUND(100.0 * COUNT(DISTINCT e.user_id) / COUNT(DISTINCT u.user_id), 2) AS pct_d28
FROM users u
LEFT JOIN events e
       ON e.user_id = u.user_id
      AND e.event_date = DATE_ADD(u.signup_date, 28)
GROUP BY u.signup_date
ORDER BY u.signup_date
"""

expect("D28 retention by cohort", SQL, [
    ("2026-01-01", 3, 0, 0.00),
    ("2026-01-02", 3, 1, 33.33),
    ("2026-01-08", 2, 0, 0.00),
])


ConnectionRefusedError: [Errno 61] Connection refused

In [16]:
"""
Problem 03: D28 retention by signup cohort.

Meta flavor: "D28 retention — does the product hold users for a month?"


"""
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()


def expect(title, sql, expected_rows):
    """Run a query and assert its exact rows, in order. Decimal/float safe."""
    import decimal

    def norm(v):
        if isinstance(v, decimal.Decimal):
            return float(v)
        if isinstance(v, float):
            return round(v, 6)
        return v

    got = [tuple(norm(c) for c in r) for r in spark.sql(sql).collect()]
    exp = [tuple(norm(c) for c in r) for r in expected_rows]
    if got != exp:
        print(f"[FAIL] {title}")
        print(f"   expected: {exp}")
        print(f"   got:      {got}")
        raise AssertionError(title)
    print(f"[PASS] {title}")
    return got


# ---------------------------------------------------------- sample data
spark.createDataFrame(
    [
    (1, "2026-01-01", "organic",  "US"),
    (2, "2026-01-01", "paid",     "US"),
    (3, "2026-01-01", "organic",  "IN"),
    (4, "2026-01-02", "paid",     "US"),
    (5, "2026-01-02", "referral", "BR"),
    (6, "2026-01-02", "organic",  "IN"),
    (7, "2026-01-08", "paid",     "US"),
    (8, "2026-01-08", "organic",  "BR"),
],
    ["user_id", "signup_date", "channel", "country"]
).createOrReplaceTempView("users")

spark.createDataFrame(
    [
    (1, "2026-01-01", "open"),  (1, "2026-01-02", "open"),  (1, "2026-01-08", "open"),
    (2, "2026-01-01", "open"),
    (3, "2026-01-01", "open"),  (3, "2026-01-02", "open"),
    (4, "2026-01-02", "open"),  (4, "2026-01-03", "open"),  (4, "2026-01-09", "open"),
    (5, "2026-01-02", "open"),
    (6, "2026-01-02", "open"),  (6, "2026-01-03", "open"),  (6, "2026-01-30", "open"),
    (7, "2026-01-08", "open"),
    (8, "2026-01-08", "open"),  (8, "2026-01-09", "open"),
],
    ["user_id", "event_date", "event_name"]
).createOrReplaceTempView("events")


SQL = """
SELECT u.signup_date AS cohort,
       COUNT(DISTINCT u.user_id) AS cohort_size,
       COUNT(DISTINCT e.user_id) AS retained_d28,
       ROUND(100.0 * COUNT(DISTINCT e.user_id) / COUNT(DISTINCT u.user_id), 2) AS pct_d28
FROM users u
LEFT JOIN events e
       ON e.user_id = u.user_id
      AND e.event_date = DATE_ADD(u.signup_date, 28)
GROUP BY u.signup_date
ORDER BY u.signup_date
"""

expect("D28 retention by cohort", SQL, [
    ("2026-01-01", 3, 0, 0.00),
    ("2026-01-02", 3, 1, 33.33),
    ("2026-01-08", 2, 0, 0.00),
])


ConnectionRefusedError: [Errno 61] Connection refused